# Outcome Simulado — Modelo Supervisionado vs. Heurística (ADR-0007)

**O que este notebook é:** um experimento controlado, não uma validação do sistema real. Nunca existiu outcome de conversão real de cliente offshore disponível publicamente (ver [ADR-0004](../docs/adr/0004-dataset-sintetico-como-premissa-consciente.md)) — então simulamos um, com ruído deliberado, para testar se um classificador supervisionado bate a heurística atual (`score_gap_total`, ver [ADR-0002](../docs/adr/0002-pesos-do-score-sao-heuristica-nao-calibracao.md)) **no próprio outcome simulado**.

**Contrato do experimento (fixado antes de rodar, não reajustado depois do resultado — ver [ADR-0007](../docs/adr/0007-outcome-simulado-para-modelo-supervisionado.md)):**
1. O outcome `converteu` é gerado com AUC teórico entre **0,75 e 0,80** contra o próprio score que o gerou — nem determinístico (o classificador só reaprenderia a fórmula), nem ruído puro (não haveria sinal nenhum pra aprender).
2. Métrica única de comparação: **AUC-ROC**, mesma base de teste (holdout), para heurística e classificador.
3. **O resultado é publicado como vier.** Sucesso não é o classificador bater a heurística — é o experimento existir, rodar de ponta a ponta e reportar o número honesto. Amarrar sucesso à vitória do modelo criaria o mesmo incentivo de resultado desenhado que motivou toda a auditoria anterior deste projeto (ADR-0001 a 0004).

**Não fazer:** reajustar o parâmetro de ruído depois de ver o resultado. Se o classificador perder, isso é achado, não falha do notebook.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

df = pd.read_csv("../data/processed/base_offshore_scored.csv")
print(f"Base carregada: {len(df):,} clientes, {df.shape[1]} colunas")
df[["id_cliente", "score_gap_total", "pct_offshore", "pct_cdi", "dias_sem_remessa"]].head()

Base carregada: 3,000 clientes, 17 colunas


,id_cliente,score_gap_total,pct_offshore,pct_cdi,dias_sem_remessa
0,G1218,80.25,0.0149,0.8066,60
1,G0138,78.75,0.0000,0.7982,60
2,G0105,77.25,0.0000,0.7091,191
3,G1196,77.00,0.0000,0.7294,58
4,G2546,76.50,0.0000,0.6116,52


## 1. Split antes de tudo

Critério de implementação do ADR-0007 (achado de auditoria PAVC): o holdout treino/teste é feito **antes** de calcular qualquer estatística (média, desvio, parâmetro de ruído) — normalizar com a base inteira e só depois splitar vazaria informação do teste pro treino, mesmo em outcome simulado.

In [2]:
FEATURES = [
    "pct_offshore", "pct_cdi", "saldo_conta_global_usd", "caixa_parado_usd",
    "dias_sem_remessa", "dolar_medio_compra", "pl_brl",
]
# Nota: usamos as features numéricas contínuas disponíveis na base já scoreada.
# São um subconjunto observável das 10 usadas no score heurístico (algumas
# sub-notas do score_gap_total não têm coluna bruta correspondente na base
# exportada — o classificador aprende sobre o que está disponível, igual
# aconteceria com qualquer feature store real).

train_idx, test_idx = train_test_split(df.index, test_size=0.3, random_state=42)
df_train = df.loc[train_idx].copy()
df_test = df.loc[test_idx].copy()
print(f"Treino: {len(df_train):,} · Teste: {len(df_test):,}")

Treino: 2,100 · Teste: 900


## 2. Gerar o outcome simulado (`converteu`)

`P(converteu) = sigmoide(score_normalizado / temperatura + ruído residual)`. A temperatura controla o quão extremas ficam as probabilidades — mesmo com ruído zero, um sorteio Bernoulli sobre probabilidade moderada (perto de 0,5) já produz um teto de AUC bem abaixo de 1,0 (achado ao calibrar: com `score_z` puro sem escala, o teto natural já ficava perto de 0,75). A temperatura é calibrada por busca binária até o AUC teórico do gerador (contra o próprio score) cair na faixa [0,75, 0,80] fixada no ADR-0007 — calculado **só no conjunto de treino**, aplicado igual nos dois splits.

In [3]:
def gerar_outcome_simulado(score, temperatura, rng, ruido_std=0.01):
    """Gera outcome binario 'converteu' a partir do score, com ruido gaussiano.

    P(converteu) = sigmoide(score_z / temperatura + ruido), onde score_z e o
    score normalizado (z-score). temperatura controla o quao extremas ficam
    as probabilidades: temperatura alta -> p_conversao fica proxima de 0.5
    pra quase todo mundo, e mesmo sem ruido nenhum o sorteio Bernoulli
    introduz um teto de AUC bem abaixo de 1.0 (probabilidade moderada =
    sorteio ainda incerto). temperatura baixa -> p_conversao vai pra perto
    de 0/1 nos extremos, sorteio fica quase deterministico, AUC sobe.
    ruido_std e um ruido residual pequeno e fixo, so pra evitar empates
    exatos na ordenacao.
    """
    score_z = (score - score.mean()) / score.std()
    ruido = rng.normal(0, ruido_std, size=len(score))
    logit = score_z / temperatura + ruido
    p_conversao = 1 / (1 + np.exp(-logit))
    converteu = rng.binomial(1, p_conversao)
    return converteu, p_conversao


def auc_teorico(score_train, temperatura, seed=42):
    rng = np.random.default_rng(seed)
    converteu, _ = gerar_outcome_simulado(score_train, temperatura, rng)
    if converteu.sum() == 0 or converteu.sum() == len(converteu):
        return None  # classe degenerada, ver checagem abaixo
    return roc_auc_score(converteu, score_train)


# Busca binaria na temperatura ate o AUC teorico cair em [0.75, 0.80],
# calculada SO no conjunto de treino (split ja feito na celula anterior).
# temperatura baixa = AUC alto (p_conversao mais extrema); temperatura
# alta = AUC baixo (p_conversao mais proxima de 0.5, sorteio mais incerto).
score_train = df_train["score_gap_total"].values
lo, hi = 0.05, 3.0
for _ in range(60):
    mid = (lo + hi) / 2
    auc = auc_teorico(score_train, mid)
    if auc is None:
        lo = mid  # degenerada -> subir temperatura (reduzir extremidade)
        continue
    if auc > 0.80:
        lo = mid  # AUC alto demais -> subir temperatura
    elif auc < 0.75:
        hi = mid  # AUC baixo demais -> baixar temperatura
    else:
        break

TEMPERATURA = mid
AUC_TEORICO = auc_teorico(score_train, TEMPERATURA)
print(f"temperatura calibrada: {TEMPERATURA:.4f}")
print(f"AUC teorico do gerador (treino): {AUC_TEORICO:.4f}  (alvo: 0.75-0.80)")
assert 0.75 <= AUC_TEORICO <= 0.80, "Calibracao fora da faixa do ADR-0007 - nao prosseguir sem revisar."

temperatura calibrada: 0.7875
AUC teorico do gerador (treino): 0.7895  (alvo: 0.75-0.80)


In [4]:
rng_train = np.random.default_rng(42)
rng_test = np.random.default_rng(43)  # seed diferente - teste nao reusa o sorteio do treino

df_train["converteu"], df_train["p_conversao"] = gerar_outcome_simulado(
    df_train["score_gap_total"].values, TEMPERATURA, rng_train
)
df_test["converteu"], df_test["p_conversao"] = gerar_outcome_simulado(
    df_test["score_gap_total"].values, TEMPERATURA, rng_test
)

# Checagem de classe unica (criterio de implementacao do ADR-0007, achado PAVC)
for nome, sub in [("treino", df_train), ("teste", df_test)]:
    n_pos = sub["converteu"].sum()
    n_neg = len(sub) - n_pos
    assert n_pos > 0 and n_neg > 0, f"Classe unica em {nome} - abortar, nao treinar sobre isso."
    print(f"{nome}: {n_pos:,} converteram ({n_pos/len(sub):.1%}) - {n_neg:,} nao converteram")

treino: 1,022 converteram (48.7%) - 1,078 nao converteram
teste: 440 converteram (48.9%) - 460 nao converteram


## 3. Treinar o classificador

Importante: o classificador aprende sobre as **features brutas** (`FEATURES`), nunca sobre `score_gap_total` diretamente — senão o experimento vira o classificador reaprendendo a própria fórmula do score, não uma comparação justa entre heurística e modelo. O `StandardScaler` é ajustado (`fit`) só no treino, aplicado (`transform`) nos dois — sem vazar estatística do teste.

In [5]:
scaler = StandardScaler()
X_train = scaler.fit_transform(df_train[FEATURES])
X_test = scaler.transform(df_test[FEATURES])

y_train = df_train["converteu"].values
y_test = df_test["converteu"].values

clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(X_train, y_train)

y_pred_proba = clf.predict_proba(X_test)[:, 1]
print("Classificador treinado (LogisticRegression, features brutas).")

Classificador treinado (LogisticRegression, features brutas).


## 4. Comparação — resultado publicado como vier

Mesma base de teste (holdout), mesmo `converteu` simulado, AUC-ROC para os dois lados. Sem reajustar `TEMPERATURA` depois deste resultado (contrato do ADR-0007).

In [6]:
auc_heuristica = roc_auc_score(y_test, df_test["score_gap_total"])
auc_modelo = roc_auc_score(y_test, y_pred_proba)

print(f"AUC-ROC - heuristica (score_gap_total): {auc_heuristica:.4f}")
print(f"AUC-ROC - classificador (LogisticRegression): {auc_modelo:.4f}")
print()
if auc_modelo > auc_heuristica:
    print(f"Resultado: classificador SUPERA a heuristica por {auc_modelo - auc_heuristica:.4f} AUC.")
elif auc_modelo < auc_heuristica:
    print(f"Resultado: classificador PERDE da heuristica por {auc_heuristica - auc_modelo:.4f} AUC.")
else:
    print("Resultado: empate tecnico.")
print()
print("Este resultado e publicado como saiu - nao houve reajuste de TEMPERATURA apos ve-lo (ADR-0007).")

AUC-ROC - heuristica (score_gap_total): 0.7753
AUC-ROC - classificador (LogisticRegression): 0.7608

Resultado: classificador PERDE da heuristica por 0.0146 AUC.

Este resultado e publicado como saiu - nao houve reajuste de TEMPERATURA apos ve-lo (ADR-0007).
